# 01. Perfilamiento de Datos
## Objetivo

Realizar una revisión inicial de las fuentes entregadas para identificar su estructura, calidad y consistencia antes de aplicar transformaciones.

El análisis incluye:

- dimensiones y esquemas;
- valores nulos;
- duplicados;
- cardinalidad de variables categóricas;
- consistencia de fechas;
- posibles anomalías de calidad;
- relaciones básicas entre las fuentes.

Los datos utilizados corresponden a la capa **Bronze** y se mantienen sin modificaciones.

## 0. Configuracion

In [1]:
### Imports

from pathlib import Path

import pandas as pd

In [2]:
### Rutas

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

raw_data_dir = project_root / "data" / "raw"

raw_data_dir

WindowsPath('c:/Users/USUARIO/Desktop/Jean/Proyectos/BG/testPeiGo/data/raw')

In [3]:
### Cargar datos

customers = pd.read_parquet(raw_data_dir / "clientes.parquet")
cards = pd.read_parquet(raw_data_dir / "tarjetas.parquet")
transactions = pd.read_parquet(raw_data_dir / "transacciones.parquet")
marketing_interactions = pd.read_parquet(raw_data_dir / "interacciones_marketing.parquet")
merchant_catalog = pd.read_parquet(raw_data_dir / "catalogo_comercios.parquet")

## 1. Revision Estructural

In [4]:
### Diccionario

datasets = {
    "customers": customers,
    "cards": cards,
    "transactions": transactions,
    "marketing_interactions": marketing_interactions,
    "merchant_catalog": merchant_catalog,
}

### 1.1 Vista general

In [ ]:

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    display(df.head())


CUSTOMERS
Rows: 12,048
Columns: 8


,cliente_id,cedula,nombre_completo,fecha_nacimiento,ciudad,canal_adquisicion,estado_cuenta,fecha_registro
0,CHK-003169,174342225-8,Ana Mendoza Gonzalez,27/07/2005,Guayaquil,publicidad_digital,activa,2024-03-17
1,CHK-006351,1308979008,Freddy Gonzalez Mendoza,1986-10-17,Guayaquil,organico,activa,2024-03-17
2,CHK-009098,1251834072,Michelle Rodriguez Rodriguez,1998-10-16,Guayaquil,organico,inactiva,05/06/2026
3,CHK-004565,1863402060,Fernando Ortiz Guaman,1997-07-28,Cuenca,call_center,activa,2023-09-15
4,CHK-007406,1396430945,Alexander Yepez Zambrano,1989-12-13,Ambato,publicidad_digital,cerrada,2021-11-11



CARDS
Rows: 16,634
Columns: 6


,tarjeta_id,cliente_id,tipo,estado_codigo,fecha_emision,fecha_activacion
0,TRJ-004448,CHK-003226,fisica,1,2026-07-01,NaN
1,TRJ-002909,CHK-002104,virtual,1,2022-09-16,2022-09-19
2,TRJ-012662,CHK-009159,virtual,1,2026-01-13,2026-01-18
3,TRJ-006817,CHK-004955,virtual,1,2023-12-03,2023-12-06
4,TRJ-010895,CHK-007913,virtual,1,2021-01-31,05/02/2021



TRANSACTIONS
Rows: 194,173
Columns: 7


,transaccion_id,cliente_id,fecha,tipo_transaccion,monto,comercio_codigo,es_devolucion
0,TX-00095123,CHK-005914,2024-01-27,cash_in,24.96,NaN,False
1,TX-00191217,CHK-011893,17/02/2023,p2p_out,11.59,NaN,False
2,TX-00068905,CHK-004297,2025-08-05,cash_in,13.71,NaN,Si
3,TX-00161080,CHK-010013,25/08/2025,P2P_OUT,18.94,NaN,False
4,TX-00079264,CHK-004942,07/06/2025,p2p_in,23.90,NaN,False



MARKETING_INTERACTIONS
Rows: 36,000
Columns: 6


,interaccion_id,cliente_id,campana,fecha_contacto,canal,respondio
0,MKT-0000001,CHK-004008,Cashback Verano,2023-07-18,email,0
1,MKT-0000002,CHK-009657,BIENVENIDA NUEVOS USUARIOS,2022-04-07,WHATSAPP,False
2,MKT-0000003,CHK-008323,REACTIVACION AHORROS,2025-10-12,sms,nan
3,MKT-0000004,CHK-004872,Cashback Verano,20/08/2024,push,False
4,MKT-0000005,CHK-008956,Reactivacion Ahorros,09/03/2023,sms,False



MERCHANT_CATALOG
Rows: 42
Columns: 3


,comercio_codigo,nombre_comercio,categoria
0,COM-001,Supermaxi,retail
1,COM-002,Mi Comisariato,salud
2,COM-003,Netflix,restaurante
3,COM-004,Spotify,supermercado
4,COM-005,Uber,streaming


#### Observaciones

La revisión visual permite identificar la granularidad y función principal de cada fuente:

- **Customers:** 12,048 clientes y 8 variables. Corresponde a la tabla maestra de clientes.
- **Cards:** 16,634 tarjetas y 6 variables. La cantidad de registros superior al número de clientes sugiere una relación uno-a-muchos entre cliente y tarjeta, que deberá validarse.
- **Transactions:** 194,173 transacciones y 7 variables. Corresponde a la principal tabla transaccional y presenta múltiples registros por cliente.
- **Marketing interactions:** 36,000 interacciones y 6 variables. Registra contactos de campañas y respuestas por cliente.
- **Merchant catalog:** 42 comercios y 3 variables. Funciona como catálogo de referencia para enriquecer las transacciones asociadas a comercios.

A nivel visual se observan posibles problemas de calidad que se validarán en las siguientes secciones:

- formatos de fecha no homogéneos;
- diferencias de mayúsculas/minúsculas en variables categóricas;
- distintas representaciones de variables booleanas;
- valores faltantes en fechas de activación de tarjetas;
- posibles inconsistencias semánticas en las categorías del catálogo de comercios.

### 1.2 Perfil estructural

In [10]:
missing_tokens = {"nan", "null", "none", ""}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(name.upper())
    print(f"{'=' * 50}")

    pseudo_missing = df.apply(
        lambda col: (
            col.astype("string")
            .str.strip()
            .str.lower()
            .isin(missing_tokens)
            .fillna(False)
            .sum()
        )
    )

    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_values": df.isna().sum(),
        "pseudo_missing": pseudo_missing,
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    display(summary)

    print(f"Exact duplicates: {df.duplicated().sum():,}")


CUSTOMERS


,dtype,missing_values,pseudo_missing,missing_pct,n_unique
cliente_id,str,0,0,0.00,12000
cedula,str,0,0,0.00,11940
nombre_completo,str,0,0,0.00,7616
fecha_nacimiento,str,0,0,0.00,10331
ciudad,str,0,0,0.00,45
canal_adquisicion,str,212,421,1.76,19
estado_cuenta,str,0,0,0.00,16
fecha_registro,str,0,0,0.00,4111


Exact duplicates: 48

CARDS


,dtype,missing_values,pseudo_missing,missing_pct,n_unique
tarjeta_id,str,0,0,0.00,16585
cliente_id,str,0,0,0.00,11691
tipo,str,0,0,0.00,11
estado_codigo,int64,0,0,0.00,3
fecha_emision,str,8,0,0.05,4047
fecha_activacion,str,2817,0,16.94,3986


Exact duplicates: 49

TRANSACTIONS


...

Exact duplicates: 1,158

MARKETING_INTERACTIONS


,dtype,missing_values,pseudo_missing,missing_pct,n_unique
interaccion_id,str,0,0,0.0,36000
cliente_id,str,0,0,0.0,11769
campana,str,0,0,0.0,15
fecha_contacto,str,0,0,0.0,5178
canal,str,0,0,0.0,8
respondio,str,0,2004,0.0,9


Exact duplicates: 0

MERCHANT_CATALOG


,dtype,missing_values,pseudo_missing,missing_pct,n_unique
comercio_codigo,str,0,0,0.0,42
nombre_comercio,str,0,0,0.0,42
categoria,str,0,0,0.0,8


Exact duplicates: 0


#### Observaciones

El perfilado inicial evidencia los siguientes puntos relevantes:

- **Customers, Cards y Transactions presentan duplicados exactos.** La cantidad de duplicados coincide con la diferencia entre el número total de registros y la cardinalidad de sus identificadores principales, lo que sugiere que la duplicidad de las claves puede estar explicada por registros completamente repetidos. Esto se validará en la revisión de claves.

- En **Customers**, `canal_adquisicion` presenta 212 valores faltantes reconocidos por pandas (1.76%) y 421 valores adicionales almacenados como pseudo-faltantes. En conjunto, aproximadamente 5.25% de los registros requieren revisión en esta variable. Además, existen 12,000 `cliente_id` únicos pero únicamente 11,940 valores distintos de `cedula`, por lo que se deberá validar si una misma identificación está asociada a múltiples clientes.

- En **Cards**, `fecha_emision` presenta 8 valores faltantes. `fecha_activacion` presenta 2,817 valores faltantes (16.94%); estos no se considerarán inicialmente un error de calidad, ya que pueden corresponder a tarjetas que nunca fueron activadas y constituir una señal relevante para el análisis del piloto.

- En **Transactions**, `monto`, `fecha` y `es_devolucion` fueron inferidos como texto, por lo que requerirán tipificación y estandarización en Silver. `comercio_codigo` presenta 90.15% de valores faltantes; esta ausencia podría ser esperada para tipos de transacción que no involucran comercios, por lo que deberá analizarse condicionada por `tipo_transaccion`.

- **Transactions contiene 13,529 `cliente_id` distintos**, frente a 12,000 clientes únicos en Customers. Esto requiere validar la integridad referencial para identificar transacciones asociadas a clientes ausentes del maestro.

- Transactions contiene **44 códigos de comercio distintos**, mientras que Merchant Catalog contiene 42, por lo que se deberá revisar la cobertura del catálogo e identificar códigos sin correspondencia.

- En **Marketing Interactions**, `respondio` fue inferido como texto y presenta 9 valores distintos. Aunque no existen faltantes reconocidos por pandas, se detectaron **2,004 pseudo-faltantes** (5.57%), lo que confirma que parte de los valores faltantes fueron almacenados como texto y deberán homologarse en Silver.

- **Merchant Catalog** no presenta valores faltantes ni duplicados exactos y sus 42 códigos de comercio son únicos. La consistencia semántica de las categorías se revisará posteriormente.

- De forma transversal, las columnas de fecha fueron cargadas como texto y varias variables categóricas presentan más valores distintos de los esperados, lo que sugiere inconsistencias de formato, capitalización o representación que deberán analizarse antes de definir las transformaciones de Silver.

### 1.3 Validación de claves e integridad referencial

In [ ]:
# Unicidad de claves principales
# Verifica si cada identificador principal es único y si la duplicidad desaparece al eliminar registros exactos repetidos.

primary_keys = {
    "customers": "cliente_id",
    "cards": "tarjeta_id",
    "transactions": "transaccion_id",
    "marketing_interactions": "interaccion_id",
    "merchant_catalog": "comercio_codigo",
}

key_checks = []

for name, key in primary_keys.items():
    df = datasets[name]
    dedup_df = df.drop_duplicates()

    key_checks.append({
        "dataset": name,
        "primary_key": key,
        "rows": len(df),
        "unique_keys": df[key].nunique(),
        "duplicate_key_rows": df.duplicated(subset=key, keep=False).sum(),
        "duplicate_key_rows_after_exact_dedup": (
            dedup_df.duplicated(subset=key, keep=False).sum()
        ),
    })

display(pd.DataFrame(key_checks))

,dataset,primary_key,rows,unique_keys,duplicate_key_rows,duplicate_key_rows_after_exact_dedup
0,customers,cliente_id,12048,12000,96,0
1,cards,tarjeta_id,16634,16585,98,0
2,transactions,transaccion_id,194173,193015,2316,0
3,marketing_interactions,interaccion_id,36000,36000,0,0
4,merchant_catalog,comercio_codigo,42,42,0,0


In [ ]:
# Integridad referencial
# Valida que las claves foráneas de tarjetas, transacciones y marketing existan en sus tablas maestras.

customer_ids = set(customers["cliente_id"].dropna().unique())
merchant_codes = set(merchant_catalog["comercio_codigo"].dropna().unique())

referential_checks = {
    "cards_customer_orphans": len(
        set(cards["cliente_id"].dropna().unique()) - customer_ids
    ),
    "transactions_customer_orphans": len(
        set(transactions["cliente_id"].dropna().unique()) - customer_ids
    ),
    "marketing_customer_orphans": len(
        set(marketing_interactions["cliente_id"].dropna().unique()) - customer_ids
    ),
    "transaction_merchant_orphans": len(
        set(transactions["comercio_codigo"].dropna().unique()) - merchant_codes
    ),
}

display(pd.Series(referential_checks, name="orphan_keys").to_frame())

,orphan_keys
cards_customer_orphans,248
transactions_customer_orphans,1531
marketing_customer_orphans,360
transaction_merchant_orphans,2


In [15]:
# Magnitud de claves huérfanas
# Calcula qué porcentaje de identificadores únicos no tiene correspondencia en la tabla maestra.

orphan_rates = {
    "cards_customer_orphans_pct": referential_checks["cards_customer_orphans"] / cards["cliente_id"].nunique() * 100,
    "transactions_customer_orphans_pct": referential_checks["transactions_customer_orphans"] / transactions["cliente_id"].nunique() * 100,
    "marketing_customer_orphans_pct": referential_checks["marketing_customer_orphans"] / marketing_interactions["cliente_id"].nunique() * 100,
}

display(pd.Series(orphan_rates, name="orphan_pct").round(2).to_frame())

,orphan_pct
cards_customer_orphans_pct,2.12
transactions_customer_orphans_pct,11.32
marketing_customer_orphans_pct,3.06


In [16]:
# Códigos de comercio sin correspondencia
# Identifica los códigos presentes en transacciones pero ausentes del catálogo.

unknown_merchant_codes = (
    set(transactions["comercio_codigo"].dropna().unique()) - merchant_codes
)

display(sorted(unknown_merchant_codes))

['COM-998', 'COM-999']

In [ ]:
# Cédulas asociadas a múltiples clientes
# Revisa si una misma identificación está vinculada a más de un cliente_id.

customers_per_id = (
    customers.drop_duplicates()
    .groupby("cedula")["cliente_id"]
    .nunique()
)

multiple_customer_ids = customers_per_id[customers_per_id > 1]

print(f"IDs linked to multiple customers: {len(multiple_customer_ids):,}")

IDs linked to multiple customers: 60


In [17]:
# Distribución de clientes por cédula
# Revisa cuántos cliente_id distintos están asociados a cada cédula problemática.

display(multiple_customer_ids.value_counts().sort_index())

cliente_id
2    60
Name: count, dtype: int64

#### Observaciones

- Después de eliminar duplicados exactos, las claves principales de **Customers, Cards y Transactions quedan completamente únicas**, por lo que la duplicidad observada inicialmente parece corresponder a registros repetidos y no a conflictos reales de clave primaria.

- Se identificaron claves foráneas de clientes sin correspondencia en **Customers**:
  - Cards: 248 clientes huérfanos (2.12% de los clientes presentes en Cards).
  - Transactions: 1,531 clientes huérfanos (11.32%), siendo la inconsistencia referencial más relevante.
  - Marketing Interactions: 360 clientes huérfanos (3.06%).

- Se encontraron **2 códigos de comercio sin correspondencia en Merchant Catalog**: `COM-998` y `COM-999`.

- Existen **60 cédulas asociadas a más de un `cliente_id`**. En todos los casos identificados, una misma cédula está vinculada exactamente a dos clientes distintos.

Estos hallazgos deberán considerarse en la construcción de Silver, especialmente para definir el tratamiento de registros huérfanos, duplicados exactos e identificadores potencialmente duplicados.

## 2. Calidad y Consistencia

### 2.1 Categoricas, fechas y tipos

In [18]:
# Consistencia de variables categóricas
# Compara cardinalidad original vs. una normalización básica de texto para detectar variantes de escritura.

categorical_columns = {
    "customers": ["ciudad", "canal_adquisicion", "estado_cuenta"],
    "cards": ["tipo"],
    "transactions": ["tipo_transaccion", "es_devolucion"],
    "marketing_interactions": ["campana", "canal", "respondio"],
    "merchant_catalog": ["categoria"],
}

category_checks = []

for name, columns in categorical_columns.items():
    for column in columns:
        series = datasets[name][column]
        normalized = series.astype("string").str.strip().str.lower()

        category_checks.append({
            "dataset": name,
            "column": column,
            "original_unique": series.nunique(dropna=True),
            "normalized_unique": normalized.nunique(dropna=True),
        })

display(pd.DataFrame(category_checks))

,dataset,column,original_unique,normalized_unique
0,customers,ciudad,45,9
1,customers,canal_adquisicion,19,7
2,customers,estado_cuenta,16,4
3,cards,tipo,11,5
4,transactions,tipo_transaccion,32,15
5,transactions,es_devolucion,10,8
6,marketing_interactions,campana,15,5
7,marketing_interactions,canal,8,4
8,marketing_interactions,respondio,9,7
9,merchant_catalog,categoria,8,8


In [19]:
# Validación de fechas y monto
# Cuantifica valores que no pueden convertirse correctamente al tipo esperado.

date_columns = {
    "customers": ["fecha_nacimiento", "fecha_registro"],
    "cards": ["fecha_emision", "fecha_activacion"],
    "transactions": ["fecha"],
    "marketing_interactions": ["fecha_contacto"],
}

type_checks = []

for name, columns in date_columns.items():
    for column in columns:
        series = datasets[name][column]
        parsed = pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)

        type_checks.append({
            "dataset": name,
            "column": column,
            "invalid_values": (series.notna() & parsed.isna()).sum(),
        })

amount_parsed = pd.to_numeric(transactions["monto"], errors="coerce")

type_checks.append({
    "dataset": "transactions",
    "column": "monto",
    "invalid_values": (transactions["monto"].notna() & amount_parsed.isna()).sum(),
})

display(pd.DataFrame(type_checks))

,dataset,column,invalid_values
0,customers,fecha_nacimiento,594
1,customers,fecha_registro,603
2,cards,fecha_emision,835
3,cards,fecha_activacion,693
4,transactions,fecha,9702
5,marketing_interactions,fecha_contacto,1800
6,transactions,monto,23289


#### Observaciones

- La normalización básica de texto (`trim` + minúsculas) reduce considerablemente la cardinalidad de la mayoría de variables categóricas, confirmando inconsistencias principalmente asociadas a mayúsculas, minúsculas y espacios. Los casos más notorios son `ciudad` (45 → 9), `estado_cuenta` (16 → 4), `tipo_transaccion` (32 → 15) y `campana` (15 → 5).

- Algunas variables continúan presentando múltiples valores incluso después de la normalización, especialmente `es_devolucion` (8 valores) y `respondio` (7 valores). Esto indica que no se trata únicamente de diferencias de formato, sino también de distintas representaciones semánticas de valores booleanos que deberán homologarse en Silver.

- `categoria` en Merchant Catalog mantiene 8 valores antes y después de la normalización, por lo que no presenta inconsistencias relevantes de capitalización o espacios. Su consistencia semántica se evaluará por separado.

- Todas las columnas de fecha contienen valores que no pueden convertirse directamente a un tipo fecha. En la mayoría de las fuentes, los registros inválidos representan aproximadamente un 5% del dataset, lo que sugiere la presencia sistemática de formatos o valores mal codificados que deberán identificarse antes de la transformación a Silver.

- `monto` presenta 23,289 valores no convertibles a numérico, aproximadamente un 12% de Transactions, siendo el problema de tipificación más relevante identificado en esta sección.

- Las transformaciones de Silver deberán incluir homologación de categorías, estandarización de variables booleanas y conversión controlada de fechas y montos, conservando trazabilidad sobre los valores que no puedan ser convertidos.

### 2.2 Anomalías relevantes

In [20]:
# Impacto de clientes huérfanos en Transactions
# Cuantifica cuántas transacciones corresponden a clientes que no existen en Customers.

orphan_transaction_mask = ~transactions["cliente_id"].isin(customer_ids)

print(f"Orphan transaction rows: {orphan_transaction_mask.sum():,}")
print(f"Orphan transaction rows pct: {orphan_transaction_mask.mean() * 100:.2f}%")

Orphan transaction rows: 1,555
Orphan transaction rows pct: 0.80%


In [22]:
# Cédulas asociadas a múltiples clientes
# Muestra los registros involucrados para identificar si existe un patrón sistemático.

duplicate_customer_records = (
    customers.drop_duplicates()
    .loc[lambda df: df["cedula"].isin(multiple_customer_ids.index)]
    .sort_values(["cedula", "fecha_registro"])
)

display(duplicate_customer_records.head(5))

,cliente_id,cedula,nombre_completo,fecha_nacimiento,ciudad,canal_adquisicion,estado_cuenta,fecha_registro
12043,CHK-002515,1044193460,Johanna Salazar Salazar,1956-08-05,Quito,organico,activa,2024-02-21
1731,CHK-004607,1044193460,Priscila Perez Vera,16/04/1951,Manta,call_center,activa,2026-04-14
8555,CHK-010166,1053898492,David Perez Ortiz,1967-04-11,Loja,publicidad_digital,cerrada,19/02/2024
1598,CHK-001653,1053898492,Wilson Castro Chiluisa,1998-08-01,Guayaquil,,activa,2026-01-18
3663,CHK-009373,1085646270,Johanna Quinde Zambrano,1990-06-03,Cuenca,Publicidad_digital,activa,2023-07-29


In [24]:
# Comercios fuera del catálogo
# Cuantifica cuántas transacciones utilizan códigos de comercio no catalogados.

unknown_merchant_codes = set(transactions["comercio_codigo"].dropna().unique()) - merchant_codes

display(
    transactions.loc[
        transactions["comercio_codigo"].isin(unknown_merchant_codes),
        "comercio_codigo",
    ].value_counts()
)

comercio_codigo
COM-998    437
COM-999    429
Name: count, dtype: int64

In [26]:
# Valores de monto no convertibles
# Inspecciona las representaciones que impiden convertir monto directamente a numérico.

invalid_amount_values = transactions.loc[
    transactions["monto"].notna() & amount_parsed.isna(),
    "monto",
]

display(invalid_amount_values.value_counts().head(10))

monto
$15.38    14
$21.27    14
$14.26    14
$13.40    13
$17.66    13
$15.93    13
$20.32    13
$6.13     13
$14.28    13
$8.94     13
Name: count, dtype: int64

In [27]:
# Representaciones de variables booleanas
# Identifica los valores que deberán homologarse en Silver.

display(transactions["es_devolucion"].value_counts(dropna=False))
display(marketing_interactions["respondio"].value_counts(dropna=False))

es_devolucion
False    104763
N         10926
No        10837
0         10798
S         10788
Si        10774
FALSE     10716
TRUE      10676
1         10654
True       3241
Name: count, dtype: int64

respondio
False    16808
True      4684
false     2155
No        2096
1         2082
0         2071
true      2055
Sí        2045
nan       2004
Name: count, dtype: int64

In [28]:
# Valores de fecha no convertibles
# Muestra ejemplos de los valores que fallan durante la conversión.

for name, columns in date_columns.items():
    for column in columns:
        series = datasets[name][column]
        parsed = pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)

        invalid_values = series[series.notna() & parsed.isna()]

        print(f"\n{name}.{column}")
        display(invalid_values.value_counts().head(10))


customers.fecha_nacimiento


fecha_nacimiento
-479066400000    2
804772800000     2
641952000000     2
1728000000       2
908690400000     2
716752800000     2
-381542400000    2
670140000000     2
1101319200000    2
775893600000     2
Name: count, dtype: int64


customers.fecha_registro


fecha_registro
1696636800000    3
1710720000000    3
1755043200000    3
1646265600000    3
1647302400000    3
1734134400000    3
1620518400000    2
1722643200000    2
1671580800000    2
1721001600000    2
Name: count, dtype: int64


cards.fecha_emision


...

fecha_activacion
1778112000000    9
1782864000000    8
1778371200000    7
1777852800000    7
1778198400000    6
1778284800000    6
1779062400000    6
1779148800000    6
1778630400000    6
1778803200000    6
Name: count, dtype: int64


transactions.fecha


fecha
1782691200000    62
1782086400000    56
1782259200000    54
1782604800000    53
1782000000000    53
1780876800000    51
1782777600000    50
1781481600000    48
1781654400000    47
1781049600000    47
Name: count, dtype: int64


marketing_interactions.fecha_contacto


fecha_contacto
1776643200000    6
1663200000000    5
1636502400000    5
1637798400000    5
1677110400000    5
1729555200000    5
1668211200000    4
1696723200000    4
1729209600000    4
1613865600000    4
Name: count, dtype: int64

In [31]:
# Revisión semántica del catálogo de comercios
# Permite detectar asignaciones de categoría potencialmente inconsistentes.

display(
    merchant_catalog
    .sort_values(["categoria", "nombre_comercio"])
    .reset_index(drop=True).head()
)

,comercio_codigo,nombre_comercio,categoria
0,COM-016,Comercio 16,entretenimiento
1,COM-017,Comercio 17,entretenimiento
2,COM-008,De Prati,entretenimiento
3,COM-014,McDonalds,entretenimiento
4,COM-021,Comercio 21,restaurante


### Observaciones

- Los clientes huérfanos representan 1,555 transacciones, equivalentes al 0.80% de Transactions. Aunque existe una cantidad relevante de `cliente_id` sin correspondencia en Customers, su impacto sobre el volumen transaccional es reducido.

- Las 60 cédulas duplicadas corresponden a `cliente_id` diferentes y presentan diferencias en atributos como nombre, fecha de nacimiento, ciudad y fecha de registro. No existe evidencia suficiente para consolidarlas como una misma persona, por lo que `cliente_id` se mantendrá como identificador principal.

- Los códigos `COM-998` y `COM-999`, ausentes de Merchant Catalog, aparecen en 437 y 429 transacciones respectivamente. Se conservarán inicialmente como comercios no catalogados en lugar de eliminar las transacciones asociadas.

- Los valores no convertibles de `monto` corresponden principalmente a representaciones monetarias con símbolo `$`, por lo que el problema es de formato y no necesariamente de contenido. La conversión deberá normalizar estas representaciones antes de tipificar la variable como numérica.

- `es_devolucion` y `respondio` utilizan múltiples representaciones para valores booleanos (`True`, `FALSE`, `Si`, `No`, `S`, `N`, `1`, `0`, entre otras). Además, `respondio` contiene `"nan"` como pseudo-faltante. Estas variables requieren homologación explícita.

- Los valores inicialmente identificados como fechas inválidas incluyen timestamps Unix almacenados como texto, principalmente en milisegundos. La transformación de Silver deberá contemplar tanto formatos de fecha convencionales como representaciones epoch.

- Merchant Catalog no presenta inconsistencias de formato en `categoria`, pero sí se observan posibles inconsistencias semánticas entre comercios conocidos y sus categorías asignadas, por lo que esta variable deberá utilizarse con cautela y conservarse trazabilidad respecto al valor original.

## 3. Criterios de Limpieza y Estandarizacion

A partir del perfilado realizado, se establecen los siguientes criterios para la preparación de los datos:

| Elemento | Criterio |
|---|---|
| Duplicados exactos | Eliminar registros completamente duplicados. En las claves primarias evaluadas en el perfilado, esta operación resuelve las duplicidades observadas sin dejar conflictos adicionales. |
| Identificación de clientes | Mantener `cliente_id` como identificador principal. No consolidar registros únicamente por coincidencia de `cedula`. |
| Claves foráneas huérfanas | Conservar los registros y generar un flag de calidad, por ejemplo `is_orphan_customer`, para controlar posteriormente su inclusión en integraciones y análisis. |
| Valores pseudo-faltantes | Homologar representaciones como `"nan"`, `"null"`, `"none"` y cadenas vacías a valores nulos reales. |
| `respondio` | Homologar sus representaciones booleanas. Los pseudo-faltantes se mantendrán como nulos y no se interpretarán como `False`, ya que ausencia de información no equivale a una respuesta negativa. |
| Variables categóricas | Eliminar espacios adicionales y homologar capitalización en las columnas categóricas identificadas. |
| Variables booleanas | Mapear las distintas representaciones de verdadero/falso a valores booleanos consistentes. |
| Fechas | Convertir formatos convencionales y timestamps Unix en segundos o milisegundos. Los valores no recuperables permanecerán nulos. |
| Fechas faltantes o inválidas | No eliminar el registro completo. La ausencia de una fecha afectará únicamente los análisis o features que dependan específicamente de ella. |
| `fecha_activacion` | Mantener valores nulos sin imputación, dado que pueden representar tarjetas que nunca fueron activadas. |
| `monto` | Remover símbolos monetarios y convertir a tipo numérico. Los valores que continúen siendo inválidos permanecerán nulos. |
| Comercios no catalogados | Conservar transacciones con códigos sin correspondencia (`COM-998`, `COM-999`) y tratarlos como comercios no catalogados durante el enriquecimiento. |
| Categorías de comercios | Estandarizar únicamente el formato. No corregir manualmente inconsistencias semánticas sin una fuente confiable de referencia. |
| Trazabilidad | Mantener los datos originales en Bronze y utilizar flags de calidad en Silver para anomalías que puedan afectar integraciones o análisis posteriores. |